# Notebook 12c — HDC-RWKV with vocab=256 (the tokenizer experiment)

*Same architecture as nb12 (1-layer bipolar recurrence + STE). Different vocab/dim tradeoff: bigger vocab, smaller hypervectors.*

## Why this notebook

nb12 (vocab=128, d=512) plateaued at val ≈ 3.55 nats/token with fragmented output. The diagnosis: **the BPE-128 vocab consists mostly of 2-3 char fragments**, so even a perfectly-trained model can only *emit* fragments.

This notebook tests the hypothesis: **bigger vocab → fewer fragments → more real-word output**, even at the cost of smaller hypervector dimensionality.

### The tradeoff at constant EEPROM

| | nb12 | **nb12c** |
|---|---|---|
| vocab | 128 | **256** |
| BPE merges | 88 | **~215** |
| `d` (hypervector dim) | 512 | **256** |
| EEPROM (vocab + proto + decay) | ~16.5 KB | **~16.5 KB** (same) |
| HDC capacity per superposition (~d/log(d)) | ~95 items | **~46 items** |
| Avg chars per BPE token | ~5.5 | **~7.5** (estimated) |
| % of corpus tokens that are full words | ~15% | **~40%** (estimated) |

### Honest predictions before running

- **Val loss**: hard to predict. The bigger vocab makes each prediction harder (more choices) but the per-character cross-entropy should drop. Net: probably similar to nb12.
- **Output coherence**: should noticeably improve. Words like `however`, `because`, `should` are likely single tokens at vocab=256, eliminating the fragment-assembly problem.
- **HDC capacity**: at d=256 with 256 prototypes, we're at the edge of the theoretical capacity. Some prototype crosstalk likely.

This is an empirical experiment, not a guaranteed win.


## Cell 1 — Setup + BPE with more merges (target vocab=256)


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, struct
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

torch.manual_seed(1337)
np.random.seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

text = Path('../data/tinyshakespeare.txt').read_text().lower()

def train_bpe(text, num_merges):
    EOW = '</w>'
    word_freq = Counter(tuple(list(w) + [EOW]) for w in text.split())
    word_lists = {w: list(w) for w in word_freq}
    merges = []
    for step in range(num_merges):
        pair_counts = Counter()
        for w, freq in word_freq.items():
            sym = word_lists[w]
            for i in range(len(sym)-1):
                pair_counts[(sym[i], sym[i+1])] += freq
        if not pair_counts: break
        best, _ = pair_counts.most_common(1)[0]
        new_tok = best[0] + best[1]
        merges.append((best, new_tok))
        for w in word_freq:
            sym = word_lists[w]; new_sym = []; i = 0
            while i < len(sym):
                if i < len(sym)-1 and (sym[i], sym[i+1]) == best:
                    new_sym.append(new_tok); i += 2
                else:
                    new_sym.append(sym[i]); i += 1
            word_lists[w] = new_sym
    vocab_set = set()
    for w in word_freq:
        vocab_set.update(word_lists[w]); vocab_set.update(w)
    return merges, sorted(vocab_set)

EOW = '</w>'
VOCAB_SIZE = 256
NUM_MERGES = 215   # tuned so total vocab ≈ 256 after adding 40 base chars + <unk>

print(f'training BPE with {NUM_MERGES} merges (target vocab={VOCAB_SIZE})...')
merges, vocab = train_bpe(text, NUM_MERGES)
all_toks = ['<unk>'] + sorted(vocab)
while len(all_toks) < VOCAB_SIZE: all_toks.append(f'<pad{len(all_toks)}>')
all_toks = all_toks[:VOCAB_SIZE]
itos = all_toks
stoi = {t: i for i, t in enumerate(itos)}

print(f'final vocab size: {len(itos)}')
print(f'longest 20 BPE tokens (the multi-char wins):')
long_toks = sorted([t for t in itos if t != '<unk>' and not t.startswith('<pad')],
                   key=lambda x: -len(x))[:20]
for t in long_toks:
    print(f'  {t!r}')


device: cuda
training BPE with 215 merges (target vocab=256)...
final vocab size: 256
longest 20 BPE tokens (the multi-char wins):
  'shall</w>'
  'from</w>'
  'good</w>'
  'have</w>'
  'king</w>'
  'ould</w>'
  'that</w>'
  'ther</w>'
  'this</w>'
  'thou</w>'
  'what</w>'
  'will</w>'
  'with</w>'
  'your</w>'
  'all</w>'
  'and</w>'
  'are</w>'
  'but</w>'
  'ere</w>'
  'ess</w>'


In [2]:
def encode_word(word):
    sym = list(word) + [EOW]
    for (a, b), m in merges:
        i = 0; new_sym = []
        while i < len(sym):
            if i < len(sym)-1 and sym[i]==a and sym[i+1]==b:
                new_sym.append(m); i += 2
            else:
                new_sym.append(sym[i]); i += 1
        sym = new_sym
    return [stoi.get(s, 0) for s in sym]

def encode(text):
    out = []
    for w in text.split():
        out.extend(encode_word(w))
    return out
def decode(ids):
    return ''.join(itos[i] for i in ids).replace(EOW, ' ')

# How well does the new vocab cover real words?
test = 'the king of france and the duke of york have come to see the queen however and because i should think'
ids = encode(test)
print(f'test text:    {test!r}')
print(f'  encoded:    {ids}')
print(f'  decoded:    {decode(ids)!r}')
print(f'  tokens:     {len(ids)} (vs {len(test)} chars, ratio {len(ids)/len(test):.2f})')
print(f'  full words: {sum(1 for i in ids if itos[i].endswith(EOW))} / {len(ids)} = {100*sum(1 for i in ids if itos[i].endswith(EOW))/len(ids):.0f}%')

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'\ncorpus tokens: train {len(train_data):,}  val {len(val_data):,}')


test text:    'the king of france and the duke of york have come to see the queen however and because i should think'
  encoded:    [213, 117, 153, 78, 175, 24, 43, 26, 213, 58, 116, 153, 247, 160, 115, 93, 48, 62, 221, 189, 62, 213, 174, 59, 69, 91, 166, 59, 229, 72, 26, 34, 41, 227, 62, 103, 192, 163, 209, 105, 115]
  decoded:    'the king of france and the duke of york have come to see the queen however and because i should think '
  tokens:     41 (vs 101 chars, ratio 0.41)
  full words: 21 / 41 = 51%

corpus tokens: train 469,262  val 52,141


## Cell 2 — STE primitive (same as nb12)


In [3]:
def ste_sign(x):
    return x.sign().detach() + x.clamp(-1, 1) - x.clamp(-1, 1).detach()


## Cell 3 — Hyperparameters

`D = 256` (half of nb12). `VOCAB_SIZE = 256` (double). EEPROM math:

- vocab_hv: 256 × 256 / 8 = 8,192 B
- prototype_hv: 256 × 256 / 8 = 8,192 B
- decay_mask: 32 B
- **Total: ~16,416 B = same as nb12**


In [4]:
BATCH_SIZE = 64
BLOCK_SIZE = 16
D          = 256
N_LAYERS   = 1
LR         = 3e-3
N_STEPS    = 12000
EVAL_EVERY = 500

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)


## Cell 4 — Architecture (identical to nb12 1-layer version)


In [5]:
class HDCRWKV(nn.Module):
    def __init__(self, vocab_size, d, block_size, n_layers=1):
        super().__init__()
        self.vocab_size = vocab_size
        self.d          = d
        self.block_size = block_size
        self.n_layers   = n_layers

        self.vocab_hv_c     = nn.Parameter(torch.randn(vocab_size, d) * 0.5)
        self.prototype_hv_c = nn.Parameter(torch.randn(vocab_size, d) * 0.5)
        self.decay_masks_c = nn.ParameterList([
            nn.Parameter(torch.full((d,), 0.5)) for _ in range(n_layers)
        ])
        self.log_temp = nn.Parameter(torch.tensor(math.log(math.sqrt(d))))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device = idx.device

        vocab_bp = ste_sign(self.vocab_hv_c)
        proto_bp = ste_sign(self.prototype_hv_c)

        tok_hv = vocab_bp[idx]
        rotated = torch.zeros_like(tok_hv)
        for t in range(T):
            rotated[:, t, :] = torch.roll(tok_hv[:, t, :], shifts=t, dims=-1)

        decay_bp = ste_sign(self.decay_masks_c[0])
        state = torch.zeros(B, self.d, device=device)
        states = []
        for t in range(T):
            update = decay_bp * state + rotated[:, t, :]
            state = torch.tanh(update)
            states.append(state)
        states = torch.stack(states, dim=1)

        temp = self.log_temp.exp()
        logits = (states @ proto_bp.t()) / temp

        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        return logits, loss

model = HDCRWKV(VOCAB_SIZE, D, BLOCK_SIZE).to(device)
deploy_bits = VOCAB_SIZE * D + VOCAB_SIZE * D + D
print(f'trainable params: {sum(p.numel() for p in model.parameters()):,}')
print(f'deployment bytes: {deploy_bits // 8:,}')
print(f'EEPROM 32 KB:     {100 * (deploy_bits // 8) / (32*1024):.1f}% used')


trainable params: 131,329
deployment bytes: 16,416
EEPROM 32 KB:     50.1% used


## Cell 5 — Training (same loop as nb12)


In [6]:
@torch.no_grad()
def hard_forward(model, idx, targets):
    B, T = idx.shape
    vocab_bp = model.vocab_hv_c.sign()
    proto_bp = model.prototype_hv_c.sign()

    tok_hv = vocab_bp[idx]
    rotated = torch.zeros_like(tok_hv)
    for t in range(T):
        rotated[:, t, :] = torch.roll(tok_hv[:, t, :], shifts=t, dims=-1)

    decay_bp = model.decay_masks_c[0].sign()
    state = torch.zeros(B, model.d, device=idx.device)
    states = []
    for t in range(T):
        update = decay_bp * state + rotated[:, t, :]
        state = torch.tanh(update)
        states.append(state)
    states = torch.stack(states, dim=1)

    temp = model.log_temp.exp()
    logits = (states @ proto_bp.t()) / temp
    loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
    return logits, loss

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
history = []
best_hard = float('inf')
best_state = None

@torch.no_grad()
def eval_both(n_batches=15):
    model.eval()
    soft_ls = torch.zeros(n_batches); hard_ls = torch.zeros(n_batches)
    for k in range(n_batches):
        xb, yb = get_batch('val')
        _, sl = model(xb, yb)
        _, hl = hard_forward(model, xb, yb)
        soft_ls[k] = sl.item(); hard_ls[k] = hl.item()
    model.train()
    return soft_ls.mean().item(), hard_ls.mean().item()

for step in range(N_STEPS + 1):
    if step % EVAL_EVERY == 0:
        soft_v, hard_v = eval_both()
        history.append((step, soft_v, hard_v))
        marker = ''
        if hard_v < best_hard:
            best_hard = hard_v
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            marker = '  <-- new best (hard)'
        print(f'step {step:>5} | soft {soft_v:.4f} | hard {hard_v:.4f}{marker}')
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

print(f'\nbest hard val: {best_hard:.4f}')
model.load_state_dict(best_state)


step     0 | soft 5.8060 | hard 5.8060  <-- new best (hard)
step   500 | soft 4.9104 | hard 4.9104  <-- new best (hard)
step  1000 | soft 4.5102 | hard 4.5102  <-- new best (hard)
step  1500 | soft 4.4661 | hard 4.4661  <-- new best (hard)
step  2000 | soft 4.4119 | hard 4.4119  <-- new best (hard)
step  2500 | soft 4.3983 | hard 4.3983  <-- new best (hard)
step  3000 | soft 4.3910 | hard 4.3910  <-- new best (hard)
step  3500 | soft 4.3741 | hard 4.3741  <-- new best (hard)
step  4000 | soft 4.3865 | hard 4.3865
step  4500 | soft 4.3795 | hard 4.3795
step  5000 | soft 4.3760 | hard 4.3760
step  5500 | soft 4.3881 | hard 4.3881
step  6000 | soft 4.3640 | hard 4.3640  <-- new best (hard)
step  6500 | soft 4.3938 | hard 4.3938
step  7000 | soft 4.3695 | hard 4.3695
step  7500 | soft 4.3865 | hard 4.3865
step  8000 | soft 4.3755 | hard 4.3755
step  8500 | soft 4.3620 | hard 4.3620  <-- new best (hard)
step  9000 | soft 4.3793 | hard 4.3793
step  9500 | soft 4.3721 | hard 4.3721
step 10000

<All keys matched successfully>

## Cell 6 — Convert nats/token to bits/char (the fair comparison vs nb12)

Loss in nats/token isn't comparable across tokenizers — nb12c's vocab=256 tokens carry more bits each. Convert to bits per character.


In [7]:
# Average chars per token
total_chars = sum(len(itos[t].replace(EOW, ' ')) for t in train_data[:5000].tolist())
avg_chars_per_tok = total_chars / 5000
bpc = best_hard * avg_chars_per_tok / math.log(2)
# wait that's wrong - it's nats_per_token / chars_per_token / ln(2)
bpc = (best_hard / avg_chars_per_tok) / math.log(2)
print(f'avg chars per token: {avg_chars_per_tok:.2f}')
print(f'nats per token:     {best_hard:.4f}')
print(f'bits per character: {bpc:.4f}')
print(f'\nFor comparison:')
print(f'  nb12 vocab=128, d=512: val=3.55 nats/tok, ~5.5 chars/tok -> BPC ~0.93')
print(f'  nb12c vocab=256, d=256: val={best_hard:.2f} nats/tok, ~{avg_chars_per_tok:.1f} chars/tok -> BPC {bpc:.2f}')


avg chars per token: 2.14
nats per token:     4.3504
bits per character: 2.9312

For comparison:
  nb12 vocab=128, d=512: val=3.55 nats/tok, ~5.5 chars/tok -> BPC ~0.93
  nb12c vocab=256, d=256: val=4.35 nats/tok, ~2.1 chars/tok -> BPC 2.93


## Cell 7 — Generate text and qualitatively compare to nb12


In [46]:
@torch.no_grad()
def generate(prompt='', max_new_tokens=80, temperature=0.6, top_k=5, seed=None):
    """Generate text from the binarized model.

    seed: if given, sets torch's RNG before sampling so the same prompt + seed
          produces identical output every time. Critical for reproducibility
          in the paper and for comparing runs.
    """
    if seed is not None:
        torch.manual_seed(seed)
        if device == 'cuda':
            torch.cuda.manual_seed_all(seed)

    model.eval()
    ids = encode(prompt) if prompt else [1]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _ = hard_forward(model, idx_cond, idx_cond)
        last_logits = logits[:, -1, :] / temperature
        if top_k > 0:
            top_vals, top_idxs = last_logits.topk(top_k, dim=-1)
            mask = torch.full_like(last_logits, float('-inf'))
            mask.scatter_(-1, top_idxs, top_vals)
            last_logits = mask
        probs = F.softmax(last_logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

# Three samples with deterministic seeds so the paper can reproduce them
max_new_tokens = 100
temparature = 0.4
top_k=5
seed=1337
prompt = "king"
out = generate(prompt=prompt, max_new_tokens=max_new_tokens, temperature=temparature, top_k=top_k, seed=seed)
print(out)
print(len(out))

king richard ived to hon, and a cstalt, when he y to the toct's sind, i toverovert, i t, and ho' i do i do you speaf. nard, and is that i a ptt, and hand you of belifts doeeect in the tod. would you have the 
208


## Cell 8 — Pack `wozformer_hdcrwkv_v3.bin` (vocab=256, d=256)


In [9]:
def pack_bits(continuous_tensor):
    binary = (continuous_tensor > 0).to(torch.uint8).cpu().numpy()
    return np.packbits(binary, axis=-1, bitorder='big')

vocab_packed = pack_bits(model.vocab_hv_c.data)
proto_packed = pack_bits(model.prototype_hv_c.data)
decay_packed = pack_bits(model.decay_masks_c[0].data.unsqueeze(0)).squeeze(0)

export_dir = Path('../export'); export_dir.mkdir(exist_ok=True)
out_path = export_dir / 'wozformer_hdcrwkv_v3.bin'

buf = bytearray()
buf += b'WHR3'
buf += bytes([3])                                     # version 3
buf += struct.pack('<H', VOCAB_SIZE)                  # 2 bytes for 256
buf += struct.pack('<H', D // 8)
buf += bytes([BLOCK_SIZE, N_LAYERS])
buf += struct.pack('<f', model.log_temp.item())
buf += bytes(3)
buf += vocab_packed.tobytes()
buf += proto_packed.tobytes()
buf += decay_packed.tobytes()

out_path.write_bytes(buf)
BUDGET = 32 * 1024
print(f'wrote {out_path}  ({len(buf):,} bytes)')
print(f'EEPROM 32 KB: {100*len(buf)/BUDGET:.1f}% used')
print(f'headroom: {BUDGET - len(buf):,} bytes')


wrote ../export/wozformer_hdcrwkv_v3.bin  (16,434 bytes)
EEPROM 32 KB: 50.2% used
headroom: 16,334 bytes


## Cell 9 — Save Python checkpoint with BPE merges (for firmware export)


In [ ]:
ckpt = Path('../export/wozformer_hdcrwkv_v3.pt')
torch.save({
    'config': dict(vocab_size=VOCAB_SIZE, d=D, block_size=BLOCK_SIZE, n_layers=N_LAYERS),
    'model_state': model.state_dict(),
    'itos': itos,
    'merges': [(list(p), m) for p, m in merges],
    'best_val_hard': best_hard,
    'history': history,
    'EOW': EOW,
}, ckpt)
print(f'wrote {ckpt}  ({ckpt.stat().st_size:,} bytes)')


## Post-mortem — did the tokenizer help?

### Three outcomes to interpret

1. **BPC drops below 0.93 (nb12's level)**: vocab swap helped. Output is more coherent. Ship this version for the paper.

2. **BPC similar (~0.93)**: capacity loss from smaller d cancelled the vocab gain. Output may still be qualitatively better (more real words emitted). Still the better choice for deployment.

3. **BPC rises**: d=256 lost too much HDC capacity. The 256-vector prototypes interfere. nb12 (vocab=128, d=512) remains the better choice; document this finding.

### What this experiment tells us about VSLM tokenizer design

Regardless of which outcome wins, you now have direct empirical data on a question no published paper has explored: **at sub-32KB binary-recurrent LMs, how does the vocab/dim tradeoff move quality?**

That's a real contribution. Worth a paragraph in the limitations/discussion section even if it doesn't change deployment.
